# RuREBus: разведочный анализ данных (EDA)

Ноутбук строит воспроизводимую картину корпуса RuREBus:

- инвентаризирует train/test и версии разметки;
- разбирает BRAT-файлы `.txt + .ann`;
- проверяет смещения, ссылки отношений и пары файлов;
- находит дубли и создаёт очищенный manifest без изменения исходных данных;
- анализирует сущности, отношения, длины и дисбаланс классов;
- формирует train/validation split по исходным документам, а не по фрагментам `_part_N`;
- полностью показывает один компактный пример: TXT, ANN, все сущности и отношения;
- сохраняет результаты в `MyDrive/NER_RuREBus_project/eda_outputs/`.

In [ ]:
# 1. Подключение Google Drive и импорты
from google.colab import drive
drive.mount("/content/drive")

from collections import Counter
from pathlib import Path
from IPython.display import HTML, Markdown, display
import hashlib
import html
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 120)

PROJECT_DIR = Path("/content/drive/MyDrive/NER_RuREBus_project")
RUREBUS_DIR = PROJECT_DIR / "rurebus_data" / "RuREBus"
OUTPUT_DIR = PROJECT_DIR / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RUREBUS_DIR.is_dir():
    raise FileNotFoundError(f"Не найдена папка RuREBus: {RUREBUS_DIR}")

print(f"RuREBus: {RUREBUS_DIR}")
print(f"Результаты EDA: {OUTPUT_DIR}")

## Схема корпуса

Сущности: `MET`, `ECO`, `BIN`, `CMP`, `QUA`, `ACT`, `INST`, `SOC`.

Отношения состояния кодируют время и оценку: `NNG/NNT/NPS`, `PNG/PNT/PPS`, `FNG/FNT/FPS`. Дополнительно используются `GOL` (цель) и `TSK` (задача).

В BRAT сущность записана как `T1\tMET 10 25\tтекст сущности`, а отношение — как `R1\tFPS Arg1:T1 Arg2:T2`.

In [ ]:
# 2. Логические версии данных и BRAT-парсер
DATA_GROUPS = {
    "train_1": RUREBUS_DIR / "train_data" / "train_part_1" / "train_part_1",
    "train_2": RUREBUS_DIR / "train_data" / "train_part_2" / "train_part_2",
    "train_3": RUREBUS_DIR / "train_data" / "train_part_3" / "train_part_3",
    "test_raw": RUREBUS_DIR / "test_data" / "test",
    "test_ner_only": RUREBUS_DIR / "test_data" / "test_ner_only",
    "test_full": RUREBUS_DIR / "test_data" / "test_full",
}

for group_name, group_path in DATA_GROUPS.items():
    if not group_path.is_dir():
        raise FileNotFoundError(f"Не найдена группа {group_name}: {group_path}")

ENTITY_TYPES = ["MET", "ECO", "BIN", "CMP", "QUA", "ACT", "INST", "SOC"]
RELATION_TYPES = ["NNG", "NNT", "NPS", "PNG", "PNT", "PPS", "FNG", "FNT", "FPS", "GOL", "TSK"]
TRAIN_PRIORITY = {"train_1": 1, "train_2": 2, "train_3": 3}

def read_utf8(path: Path) -> str:
    # newline="" сохраняет переводы строк: BRAT-смещения считаются по исходному тексту.
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        return stream.read()

def source_document_id(stem: str) -> str:
    return re.sub(r"_part_\d+_?$", "", stem)

def text_sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def parse_brat(ann_path: Path, text: str):
    entities = {}
    relations = []
    issues = []

    for line_number, raw_line in enumerate(read_utf8(ann_path).splitlines(), start=1):
        if not raw_line.strip():
            continue
        try:
            if raw_line.startswith("T"):
                parts = raw_line.split("\t", 2)
                entity_id = parts[0]
                entity_type, span_spec = parts[1].split(" ", 1)
                surface = parts[2] if len(parts) == 3 else ""
                spans = [tuple(map(int, item.split())) for item in span_spec.split(";")]
                entities[entity_id] = {
                    "entity_id": entity_id,
                    "entity_type": entity_type,
                    "spans": spans,
                    "start": spans[0][0],
                    "end": spans[-1][1],
                    "ann_text": surface,
                }
            elif raw_line.startswith("R"):
                relation_id, spec = raw_line.split("\t", 1)
                fields = spec.split()
                args = dict(field.split(":", 1) for field in fields[1:] if ":" in field)
                relations.append({
                    "relation_id": relation_id,
                    "relation_type": fields[0],
                    "arg1": args.get("Arg1"),
                    "arg2": args.get("Arg2"),
                })
            else:
                issues.append({"kind": "unsupported_line", "line": line_number, "value": raw_line})
        except Exception as error:
            issues.append({"kind": "malformed_line", "line": line_number, "value": raw_line, "error": repr(error)})

    return entities, relations, issues

In [ ]:
# 3. Полное сканирование корпуса и проверки целостности
document_rows = []
entity_rows = []
relation_rows = []
issue_rows = []

for group_name, group_path in DATA_GROUPS.items():
    txt_files = {path.stem: path for path in group_path.glob("*.txt")}
    ann_files = {path.stem: path for path in group_path.glob("*.ann")}

    for missing_stem in sorted(set(txt_files) - set(ann_files)):
        if group_name != "test_raw":
            issue_rows.append({"group": group_name, "stem": missing_stem, "kind": "missing_ann", "details": ""})
    for missing_stem in sorted(set(ann_files) - set(txt_files)):
        issue_rows.append({"group": group_name, "stem": missing_stem, "kind": "missing_txt", "details": ""})

    for stem, txt_path in sorted(txt_files.items()):
        text = read_utf8(txt_path)
        document_key = f"{group_name}:{stem}"
        ann_path = ann_files.get(stem)
        entities, relations, parser_issues = ({}, [], [])
        if ann_path is not None:
            entities, relations, parser_issues = parse_brat(ann_path, text)

        for parser_issue in parser_issues:
            issue_rows.append({
                "group": group_name, "stem": stem,
                "kind": parser_issue["kind"],
                "details": json.dumps(parser_issue, ensure_ascii=False),
            })

        document_rows.append({
            "document_key": document_key,
            "group": group_name,
            "stem": stem,
            "source_id": source_document_id(stem),
            "txt_path": str(txt_path),
            "ann_path": str(ann_path) if ann_path else "",
            "text_sha256": text_sha256(text),
            "characters": len(text),
            "whitespace_tokens": len(text.split()),
            "entity_count": len(entities),
            "relation_count": len(relations),
        })

        for entity in entities.values():
            spans = entity["spans"]
            valid_bounds = all(0 <= start <= end <= len(text) for start, end in spans)
            actual_text = " ".join(text[start:end] for start, end in spans) if valid_bounds else ""
            text_matches = valid_bounds and actual_text == entity["ann_text"]
            entity_rows.append({
                "document_key": document_key, "group": group_name, "stem": stem,
                "source_id": source_document_id(stem),
                "entity_id": entity["entity_id"], "entity_type": entity["entity_type"],
                "start": entity["start"], "end": entity["end"],
                "spans": json.dumps(spans), "ann_text": entity["ann_text"],
                "actual_text": actual_text, "valid_bounds": valid_bounds,
                "text_matches": text_matches, "discontinuous": len(spans) > 1,
                "character_length": sum(end - start for start, end in spans),
                "word_length": len(actual_text.split()),
            })
            if not valid_bounds:
                issue_rows.append({"group": group_name, "stem": stem, "kind": "invalid_entity_bounds", "details": entity["entity_id"]})
            elif not text_matches:
                issue_rows.append({"group": group_name, "stem": stem, "kind": "entity_text_mismatch", "details": entity["entity_id"]})

        for relation in relations:
            arg1 = entities.get(relation["arg1"])
            arg2 = entities.get(relation["arg2"])
            missing_arg = arg1 is None or arg2 is None
            if missing_arg:
                issue_rows.append({"group": group_name, "stem": stem, "kind": "missing_relation_argument", "details": relation["relation_id"]})
            cover_start = min(arg1["start"], arg2["start"]) if not missing_arg else np.nan
            cover_end = max(arg1["end"], arg2["end"]) if not missing_arg else np.nan
            relation_rows.append({
                "document_key": document_key, "group": group_name, "stem": stem,
                "source_id": source_document_id(stem),
                "relation_id": relation["relation_id"], "relation_type": relation["relation_type"],
                "arg1": relation["arg1"], "arg1_type": arg1["entity_type"] if arg1 else "",
                "arg1_text": arg1["ann_text"] if arg1 else "",
                "arg2": relation["arg2"], "arg2_type": arg2["entity_type"] if arg2 else "",
                "arg2_text": arg2["ann_text"] if arg2 else "",
                "missing_argument": missing_arg,
                "cover_characters": cover_end - cover_start if not missing_arg else np.nan,
                "crosses_newline": ("\n" in text[int(cover_start):int(cover_end)]) if not missing_arg else False,
            })

documents = pd.DataFrame(document_rows)
entities = pd.DataFrame(entity_rows)
relations = pd.DataFrame(relation_rows)
issues = pd.DataFrame(issue_rows, columns=["group", "stem", "kind", "details"])

overview = documents.groupby("group").agg(
    txt_files=("stem", "size"),
    source_groups=("source_id", "nunique"),
    characters=("characters", "sum"),
    whitespace_tokens=("whitespace_tokens", "sum"),
    entities=("entity_count", "sum"),
    relations=("relation_count", "sum"),
).reset_index()
display(overview)

if issues.empty:
    print("Проверка BRAT: ошибок не обнаружено.")
else:
    display(issues.groupby(["group", "kind"]).size().rename("count").reset_index())

In [ ]:
# 4. Дубли train и очищенный набор (исходные файлы не изменяются)
train_all = documents[documents["group"].isin(TRAIN_PRIORITY)].copy()
train_all["priority"] = train_all["group"].map(TRAIN_PRIORITY)
train_all = train_all.sort_values(["priority", "group", "stem"], ascending=[False, True, True])

duplicate_hashes = train_all.groupby("text_sha256").size()
duplicate_hashes = set(duplicate_hashes[duplicate_hashes > 1].index)
duplicates = train_all[train_all["text_sha256"].isin(duplicate_hashes)].copy()
duplicates["kept"] = ~duplicates.duplicated("text_sha256", keep="first")

train_docs = train_all.drop_duplicates("text_sha256", keep="first").copy()
selected_keys = set(train_docs["document_key"])
train_entities = entities[entities["document_key"].isin(selected_keys)].copy()
train_relations = relations[relations["document_key"].isin(selected_keys)].copy()

print(f"Train до дедупликации: {len(train_all)} фрагментов")
print(f"Групп одинаковых текстов: {len(duplicate_hashes)}")
print(f"Train после дедупликации: {len(train_docs)} фрагментов")
print(f"Исходных документальных групп: {train_docs['source_id'].nunique()}")
print(f"Сущностей: {len(train_entities):,}")
print(f"Отношений: {len(train_relations):,}")

display(duplicates[["group", "stem", "source_id", "entity_count", "relation_count", "kept"]].sort_values(["source_id", "group"]))

## Распределение сущностей

Графики ниже строятся только по очищенному train. Порядок типов фиксирован, поэтому отсутствующий класс также был бы заметен.

In [ ]:
# 5. Сущности: частоты и длины
entity_counts = train_entities["entity_type"].value_counts().reindex(ENTITY_TYPES, fill_value=0)
entity_stats = pd.DataFrame({
    "count": entity_counts,
    "percent": (100 * entity_counts / entity_counts.sum()).round(2),
}).sort_values("count", ascending=False)
display(entity_stats)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=entity_stats.index, y=entity_stats["count"], ax=axes[0], color="#4C78A8")
axes[0].set(title="Сущности по типам", xlabel="Тип", ylabel="Количество")
sns.boxplot(data=train_entities, x="entity_type", y="word_length", order=ENTITY_TYPES, ax=axes[1], color="#72B7B2", showfliers=False)
axes[1].set(title="Длина сущности в словах (без выбросов)", xlabel="Тип", ylabel="Слова")
plt.tight_layout()
plt.show()

display(train_entities.groupby("entity_type")[["character_length", "word_length"]].describe(percentiles=[0.5, 0.9, 0.99]).round(2))

## Распределение отношений

`NO_RELATION` отсутствует в BRAT и должен генерироваться при подготовке кандидатов. Доля положительных связей среди всех пар показывает, почему полный перебор пар без negative sampling непрактичен.

In [ ]:
# 6. Отношения: частоты, типы аргументов и геометрия
relation_counts = train_relations["relation_type"].value_counts().reindex(RELATION_TYPES, fill_value=0)
relation_stats = pd.DataFrame({
    "count": relation_counts,
    "percent": (100 * relation_counts / relation_counts.sum()).round(2),
}).sort_values("count", ascending=False)
display(relation_stats)

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(x=relation_stats.index, y=relation_stats["count"], ax=axes[0], color="#F58518")
axes[0].set(title="Отношения по типам", xlabel="Тип", ylabel="Количество")
sns.histplot(train_relations["cover_characters"], bins=60, ax=axes[1], color="#54A24B")
axes[1].set_xlim(0, train_relations["cover_characters"].quantile(0.99))
axes[1].set(title="Расстояние, покрываемое отношением (до p99)", xlabel="Символы", ylabel="Количество")
plt.tight_layout()
plt.show()

top_pairs = (train_relations.groupby(["relation_type", "arg1_type", "arg2_type"]).size()
             .rename("count").sort_values(ascending=False).head(25).reset_index())
display(top_pairs)

possible_pairs = int((train_docs["entity_count"] * (train_docs["entity_count"] - 1) / 2).sum())
positive_fraction = len(train_relations) / possible_pairs
print(f"Все неупорядоченные пары сущностей: {possible_pairs:,}")
print(f"Положительные отношения: {len(train_relations):,}")
print(f"Доля положительных пар: {positive_fraction:.4%}")
print(f"Отношений, пересекающих перенос строки: {int(train_relations['crosses_newline'].sum()):,} из {len(train_relations):,}")
display(train_relations["cover_characters"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T.round(2))

In [ ]:
# 7. Длины фрагментов: следствие для RuBERT
display(train_docs[["characters", "whitespace_tokens", "entity_count", "relation_count"]].describe(percentiles=[0.5, 0.9, 0.99]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(train_docs["whitespace_tokens"], bins=40, ax=axes[0], color="#4C78A8")
axes[0].axvline(512, color="red", linestyle="--", label="512 токенов (ориентир)")
axes[0].legend()
axes[0].set(title="Длина фрагментов", xlabel="Слова по пробелам", ylabel="Количество")
sns.scatterplot(data=train_docs, x="whitespace_tokens", y="entity_count", hue="group", ax=axes[1])
axes[1].set(title="Длина текста и число сущностей", xlabel="Слова", ylabel="Сущности")
plt.tight_layout()
plt.show()

print("Примечание: число слов не равно числу subword-токенов RuBERT; фактическую нарезку нужно рассчитывать токенизатором модели.")

## Версии теста и защита от утечки

`test_raw` — исходный тест без меток; `test_ner_only` — 544 текста с сущностями; `test_full` — 30 текстов с сущностями и отношениями. Полный тест является подмножеством NER-теста.

In [ ]:
# 8. Сопоставление тестовых версий и безопасный дополнительный NER-корпус
test_raw = documents[documents["group"] == "test_raw"].set_index("stem")
test_ner = documents[documents["group"] == "test_ner_only"].set_index("stem")
test_full = documents[documents["group"] == "test_full"].set_index("stem")

raw_ner_common = test_raw.index.intersection(test_ner.index)
raw_ner_same_text = (test_raw.loc[raw_ner_common, "text_sha256"] == test_ner.loc[raw_ner_common, "text_sha256"]).sum()
full_ner_common = test_full.index.intersection(test_ner.index)

full_entity_keys = set(entities[entities["group"] == "test_full"]["stem"])
aux_ner_docs = documents[(documents["group"] == "test_ner_only") & (~documents["stem"].isin(full_entity_keys))].copy()
aux_keys = set(aux_ner_docs["document_key"])
aux_ner_entities = entities[entities["document_key"].isin(aux_keys)].copy()

test_version_summary = pd.DataFrame([
    {"check": "Общие имена test_raw и test_ner_only", "value": len(raw_ner_common)},
    {"check": "Полностью одинаковые тексты raw/NER", "value": int(raw_ner_same_text)},
    {"check": "test_full входит в test_ner_only", "value": len(full_ner_common)},
    {"check": "Безопасные дополнительные NER-файлы", "value": len(aux_ner_docs)},
    {"check": "Сущности в дополнительном NER-корпусе", "value": len(aux_ner_entities)},
])
display(test_version_summary)

print("Для официально сопоставимого benchmark не обучайтесь на test_ner_only.")
print("Для project-max-data режима можно использовать aux_ner_docs, сохранив 30 test_full полностью закрытыми.")

## Групповой train/validation split

Все `_part_N` одного исходного документа получают один split. Среди случайных вариантов выбирается тот, чьё распределение отношений ближе к целевым 20%, со штрафом за отсутствие редких классов.

In [ ]:
# 9. Детерминированный групповой split с приближённой стратификацией отношений
VALIDATION_FRACTION = 0.20
RANDOM_SEED = 42
N_SEARCH_TRIALS = 5000

source_ids = np.array(sorted(train_docs["source_id"].unique()))
relation_by_source = (pd.crosstab(train_relations["source_id"], train_relations["relation_type"])
                      .reindex(index=source_ids, columns=RELATION_TYPES, fill_value=0))
target_group_count = max(1, round(len(source_ids) * VALIDATION_FRACTION))
target_relation_counts = relation_by_source.sum(axis=0).to_numpy() * VALIDATION_FRACTION
total_relation_counts = relation_by_source.sum(axis=0).to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
best_score = float("inf")
best_validation_sources = None

for _ in range(N_SEARCH_TRIALS):
    candidate = rng.choice(source_ids, size=target_group_count, replace=False)
    validation_counts = relation_by_source.loc[candidate].sum(axis=0).to_numpy()
    train_counts = total_relation_counts - validation_counts
    distribution_error = np.mean(np.abs(validation_counts - target_relation_counts) / (target_relation_counts + 1))
    missing_validation = np.sum((total_relation_counts > 0) & (validation_counts == 0))
    missing_train = np.sum((total_relation_counts > 0) & (train_counts == 0))
    score = distribution_error + 2.0 * missing_validation + 5.0 * missing_train
    if score < best_score:
        best_score = score
        best_validation_sources = set(candidate)

train_docs["split"] = np.where(train_docs["source_id"].isin(best_validation_sources), "validation", "train")
train_entities = train_entities.merge(train_docs[["document_key", "split"]], on="document_key", how="left")
train_relations = train_relations.merge(train_docs[["document_key", "split"]], on="document_key", how="left")

assert set(train_docs.loc[train_docs["split"] == "train", "source_id"]).isdisjoint(
    set(train_docs.loc[train_docs["split"] == "validation", "source_id"])
)
assert set(train_docs.loc[train_docs["split"] == "train", "text_sha256"]).isdisjoint(
    set(train_docs.loc[train_docs["split"] == "validation", "text_sha256"])
)

split_summary = train_docs.groupby("split").agg(
    fragments=("document_key", "size"),
    source_groups=("source_id", "nunique"),
    entities=("entity_count", "sum"),
    relations=("relation_count", "sum"),
).reset_index()
display(split_summary)

split_relation_table = (pd.crosstab(train_relations["relation_type"], train_relations["split"])
                        .reindex(RELATION_TYPES, fill_value=0))
split_relation_table["validation_percent"] = (
    100 * split_relation_table.get("validation", 0) / split_relation_table[["train", "validation"]].sum(axis=1)
).round(2)
display(split_relation_table)
print(f"Split score: {best_score:.4f}; seed={RANDOM_SEED}")

## Один полный пример TXT + ANN

Чтобы вывод оставался читаемым, автоматически выбирается самый короткий очищенный train-фрагмент, содержащий хотя бы одно отношение. Ниже печатаются исходный TXT, исходный ANN, затем все сущности и все отношения этого примера.

In [ ]:
# 10. Полный пример: TXT, ANN, все сущности и все отношения
sample_doc = (train_docs[train_docs["relation_count"] > 0]
              .sort_values(["characters", "stem"])
              .iloc[0])
sample_key = sample_doc["document_key"]
sample_text = read_utf8(Path(sample_doc["txt_path"]))
sample_ann = read_utf8(Path(sample_doc["ann_path"]))

display(Markdown(
    f"**Документ:** `{sample_doc['stem']}`  \n"
    f"**Партия:** `{sample_doc['group']}`  \n"
    f"**Сущностей:** {sample_doc['entity_count']}; **отношений:** {sample_doc['relation_count']}"
))

display(Markdown("### Полный TXT"))
display(HTML(f"<pre style='white-space:pre-wrap;max-height:500px;overflow:auto'>{html.escape(sample_text)}</pre>"))
display(Markdown("### Полный ANN"))
display(HTML(f"<pre style='white-space:pre-wrap;max-height:500px;overflow:auto'>{html.escape(sample_ann)}</pre>"))

sample_entities = (train_entities[train_entities["document_key"] == sample_key]
                   [["entity_id", "entity_type", "start", "end", "ann_text"]]
                   .sort_values(["start", "end"]).reset_index(drop=True))
sample_relations = (train_relations[train_relations["document_key"] == sample_key]
                    [["relation_id", "relation_type", "arg1", "arg1_type", "arg1_text",
                      "arg2", "arg2_type", "arg2_text", "cover_characters"]]
                    .sort_values("relation_id").reset_index(drop=True))

display(Markdown("### Все сущности примера"))
with pd.option_context("display.max_rows", None):
    display(sample_entities)

display(Markdown("### Все отношения между сущностями примера"))
with pd.option_context("display.max_rows", None):
    display(sample_relations)

## Сохранение результатов

Сохраняются только производные таблицы. Исходные `.txt`, `.ann` и ZIP-файлы не изменяются.

In [ ]:
# 11. Экспорт manifest и таблиц EDA
artifacts = {
    "dataset_overview.csv": overview,
    "brat_issues.csv": issues,
    "train_duplicates.csv": duplicates,
    "train_documents_deduplicated.csv": train_docs,
    "train_entities.csv": train_entities,
    "train_relations.csv": train_relations,
    "auxiliary_ner_documents.csv": aux_ner_docs.reset_index(drop=True),
    "auxiliary_ner_entities.csv": aux_ner_entities,
    "test_version_summary.csv": test_version_summary,
}

for filename, dataframe in artifacts.items():
    output_path = OUTPUT_DIR / filename
    dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{filename}: {output_path.stat().st_size / 1024**2:.2f} МБ")

metadata = {
    "random_seed": RANDOM_SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "search_trials": N_SEARCH_TRIALS,
    "train_fragments_before_deduplication": len(train_all),
    "train_fragments_after_deduplication": len(train_docs),
    "train_source_groups": int(train_docs["source_id"].nunique()),
    "train_entities": len(train_entities),
    "train_relations": len(train_relations),
    "test_full_documents": len(test_full),
    "auxiliary_ner_documents": len(aux_ner_docs),
}
metadata_path = OUTPUT_DIR / "eda_metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"eda_metadata.json: {metadata_path}")

## Как интерпретировать результат

- Для воспроизводимого NER+RE baseline используйте `train_documents_deduplicated.csv` и колонку `split`.
- `test_full` остаётся закрытым итоговым тестом.
- `auxiliary_ner_*` можно подключать только в отдельном project-max-data эксперименте.
- Для RuBERT потребуется нарезка на предложения или перекрывающиеся окна.
- Для RE генерируйте кандидатов локально и применяйте negative sampling: полный перебор создаёт подавляющее большинство `NO_RELATION`.